In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import random

# ============================================================
# REPLAY BUFFER
# ============================================================

class ReplayBuffer:
    def __init__(self, max_size=1000):
        self.max_size = max_size
        self.buffer = []

    def add(self, x, y, task_id):
        for i in range(len(x)):
            if len(self.buffer) >= self.max_size:
                self.buffer.pop(0)

            self.buffer.append((x[i], y[i], task_id))

    def sample(self, batch_size):
        samples = random.sample(
            self.buffer,
            min(batch_size, len(self.buffer))
        )

        x = np.array([s[0] for s in samples])
        y = np.array([s[1] for s in samples])
        t = np.array([s[2] for s in samples])

        return x, y, t


# ============================================================
# SHARED FEATURE EXTRACTOR
# ============================================================

def build_shared_encoder(input_shape=(128,128,3)):

    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    model = models.Model(inputs, x, name="Shared_Encoder")

    return model


# ============================================================
# TASK 1 : CLASSIFICATION MODEL
# ============================================================

def build_classification_model(shared_encoder):

    inputs = layers.Input(shape=(128,128,3))

    x = shared_encoder(inputs)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    outputs = layers.Dense(4, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model


# ============================================================
# TASK 2 : SEGMENTATION MODEL
# ============================================================

def build_segmentation_model(shared_encoder):

    inputs = layers.Input(shape=(128,128,3))

    x = shared_encoder(inputs)

    x = layers.Conv2DTranspose(64, 3, strides=2,
                               padding='same',
                               activation='relu')(x)

    x = layers.Conv2DTranspose(32, 3, strides=2,
                               padding='same',
                               activation='relu')(x)

    x = layers.Conv2DTranspose(16, 3, strides=2,
                               padding='same',
                               activation='relu')(x)

    outputs = layers.Conv2D(
        1,
        1,
        activation='sigmoid'
    )(x)

    model = models.Model(inputs, outputs)

    return model


# ============================================================
# TASK 3 : PREDICTION / REGRESSION MODEL
# ============================================================

def build_prediction_model(shared_encoder):

    inputs = layers.Input(shape=(128,128,3))

    x = shared_encoder(inputs)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    outputs = layers.Dense(1, activation='linear')(x)

    model = models.Model(inputs, outputs)

    return model


# ============================================================
# BUILD MODELS
# ============================================================

shared_encoder = build_shared_encoder()

classification_model = build_classification_model(shared_encoder)
segmentation_model = build_segmentation_model(shared_encoder)
prediction_model = build_prediction_model(shared_encoder)

# ============================================================
# COMPILE MODELS
# ============================================================

classification_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

segmentation_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

prediction_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='mse',
    metrics=['mae']
)

# ============================================================
# CREATE REPLAY BUFFER
# ============================================================

replay_buffer = ReplayBuffer(max_size=2000)

# ============================================================
# DUMMY DATA
# Replace with real dataset
# ============================================================

# ----- Task 1 : Classification -----

x_class = np.random.rand(200,128,128,3)

y_class = tf.keras.utils.to_categorical(
    np.random.randint(0,4,200),
    num_classes=4
)

# ----- Task 2 : Segmentation -----

x_seg = np.random.rand(200,128,128,3)

y_seg = np.random.randint(
    0,
    2,
    (200,16,16,1)
)

# ----- Task 3 : Prediction -----

x_pred = np.random.rand(200,128,128,3)

y_pred = np.random.rand(200,1)

# ============================================================
# CONTINUAL LEARNING TRAINING
# ============================================================

EPOCHS = 5
BATCH_SIZE = 16

# ============================================================
# TASK 1 TRAINING
# ============================================================

print("\nTraining Task 1 : Classification")

classification_model.fit(
    x_class,
    y_class,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

# Store samples into replay memory
replay_buffer.add(x_class, y_class, task_id=1)

# ============================================================
# TASK 2 TRAINING WITH REPLAY
# ============================================================

print("\nTraining Task 2 : Segmentation")

# Sample replay data
rx, ry, rt = replay_buffer.sample(50)

# Combine current + replay data
combined_x = np.concatenate([x_seg, rx], axis=0)

# Create dummy segmentation labels for replay
dummy_seg_labels = np.zeros((len(rx),16,16,1))

combined_y = np.concatenate(
    [y_seg, dummy_seg_labels],
    axis=0
)

segmentation_model.fit(
    combined_x,
    combined_y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

# Store segmentation task samples
replay_buffer.add(x_seg, y_seg, task_id=2)

# ============================================================
# TASK 3 TRAINING WITH REPLAY
# ============================================================

print("\nTraining Task 3 : Prediction")

rx, ry, rt = replay_buffer.sample(50)

combined_x = np.concatenate([x_pred, rx], axis=0)

dummy_pred_labels = np.zeros((len(rx),1))

combined_y = np.concatenate(
    [y_pred, dummy_pred_labels],
    axis=0
)

prediction_model.fit(
    combined_x,
    combined_y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

# ============================================================
# EVALUATION
# ============================================================

print("\nEvaluating Classification Task")
classification_model.evaluate(x_class, y_class)

print("\nEvaluating Segmentation Task")
segmentation_model.evaluate(x_seg, y_seg)

print("\nEvaluating Prediction Task")
prediction_model.evaluate(x_pred, y_pred)

# ============================================================
# SAVE MODELS
# ============================================================

classification_model.save("classification_cl_model.h5")
segmentation_model.save("segmentation_cl_model.h5")
prediction_model.save("prediction_cl_model.h5")

print("\nContinual Learning Training Completed.")
```
